# Stage 2C checkpoint export

Copies the completed Stage-2C checkpoint to a stable neutral output filename for downstream experiments.

In [ ]:
from pathlib import Path
import shutil, torch

INPUT = Path('/kaggle/input')
required = {'fusion_state_dict', 'label_model_state_dict', 'bottle_model_state_dict', 'manual_val', 'hard_val'}
matches = []
for path in INPUT.rglob('*_best.pt'):
    try:
        payload = torch.load(path, map_location='cpu', weights_only=False, mmap=True)
        if isinstance(payload, dict) and required.issubset(payload):
            matches.append((path, int(payload.get('epoch', -1))))
        del payload
    except Exception:
        continue
if len(matches) != 1:
    raise RuntimeError(f'Expected one completed Stage-2C checkpoint; found: {matches}')
source, epoch = matches[0]
destination = Path('/kaggle/working/manual_stage2c_best.pt')
shutil.copy2(source, destination)
print({'checkpoint': str(destination), 'bytes': destination.stat().st_size, 'epoch': epoch}, flush=True)
